In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import pandas as pd
import pickle

In [2]:
#load the trained model, scaler, encoder files
model = load_model('model.h5')

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)
    
with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)
    
with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [3]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [5]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [ ]:
# One-hot encode 'Geography'
geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [7]:
## Encode categorical variables 'gender' 
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])


In [8]:
## concatination one hot encoded 
input_df = pd.concat([input_df.drop('Geography', axis = 1), geo_encoded_df], axis = 1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [10]:
input_scaled = scaler.transform(input_df)
input_scaled

c:\Users\TANVI AGARWAL\ANN Classification\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


array([[-6.7681861 ,  0.73902774, -3.68814088, -1.97778513, -1.21847468,
        -1.24715903, -0.11888692,  0.92443458, -1.74618097,  1.00450338,
        -1.91524949, -1.90861117]])

In [ ]:
## Predict churn
prediction = model.predict(input_scaled) #double bracket in output means Outer bracket  →  which customer  (row) Inner bracket  →  their result(column)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step


array([[0.00037932]], dtype=float32)

In [12]:
prediction_probability = prediction[0][0]

In [13]:
prediction_probability

np.float32(0.0003793226)

In [14]:
if prediction_probability > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
